# 02 - Entrenamiento y selección del mejor modelo
### Clasificador de Pokémon por tipo (Generación 1)

Este notebook cubre la tercera entrega del TP Integrador:
1. Carga del dataset y DataLoaders mediante `dataset_loader.py` (raíz del repo)
2. Modelo preentrenado y estrategia de fine-tuning
3. Configuración del entrenamiento
4. Experimentación (4 configuraciones distintas)
5. Evaluación del modelo elegido (curvas, métricas en test, matriz de confusión, análisis de errores)
6. Guardado del modelo final (`dev/modelo.pth`)

**Hardware utilizado:** Google Colab con GPU T4.

**Para correr este notebook:** clonar el repo, ejecutar `dev/01_remove_background.ipynb`
(o asegurarse de que `data/PokemonData/` ya esté poblado), y correr este notebook desde
la **raíz del repositorio** (la primera celda de código se encarga de pararse ahí si
detecta que se está ejecutando desde `dev/`).

## 1. Setup e imports

Reutilizamos `dataset_loader.py`, que ya define: `classes`, `label_to_idx`, `idx_to_label`,
`train_df` / `val_df` / `test_df`, `train_dataset` / `val_dataset` / `test_dataset` y
`train_loader` / `val_loader` / `test_loader`, además de las transformaciones (`Resize(256)`
+ `CenterCrop(224)` + normalización ImageNet, con augmentations solo en train).

In [ ]:
import os
import sys

# Ajustar el path si se corre desde dev/
if os.path.basename(os.getcwd()) == "dev":
    os.chdir("..")
sys.path.append(os.getcwd())

!pip install -q -r requirements.txt

In [ ]:
import json
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from dataset_loader import (
    classes,
    label_to_idx,
    idx_to_label,
    train_df,
    val_df,
    test_df,
    train_dataset,
    val_dataset,
    test_dataset,
    train_loader,
    val_loader,
    test_loader,
    BATCH_SIZE,
    SEED,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(classes)

print("\nDispositivo utilizado:", device)
print("Cantidad de clases:", num_classes)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## 2. Modelo preentrenado y estrategia de fine-tuning

**Modelo elegido: ResNet18 preentrenada en ImageNet.**

Justificación: ResNet18 es un modelo liviano (~11M parámetros), rápido de entrenar en Colab
y suficientemente potente para un dataset relativamente chico como el nuestro (~6800
imágenes, 15 clases). Modelos más grandes (ResNet50, EfficientNet-B4, ViT) tienen muchos
más parámetros y serían más propensos a overfitting con esta cantidad de datos, además de
requerir más tiempo de entrenamiento.

**Modificaciones a la arquitectura:**
- Se reemplazó la capa final `fc` (originalmente 1000 salidas para ImageNet) por una capa
  `nn.Linear` con `num_classes` (15) salidas, una por cada tipo de Pokémon.

**Estrategia de fine-tuning:**
- Se **congelan todas las capas convolucionales preentrenadas** (`requires_grad=False`).
  Estas capas ya aprendieron filtros generales (bordes, texturas, colores, formas) en
  ImageNet que son útiles también para imágenes de Pokémon.
- Solo se entrena la **capa final (`fc`)** desde cero.
- Esta decisión se toma porque el dataset es relativamente chico: entrenar toda la red
  end-to-end aumentaría mucho el riesgo de overfitting y el tiempo de entrenamiento, sin
  necesariamente mejorar el resultado dado que el dominio (imágenes de criaturas/dibujos)
  no es tan distinto del dominio de ImageNet.

In [ ]:
def crear_modelo_resnet18():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Congelar todas las capas preentrenadas
    for param in model.parameters():
        param.requires_grad = False

    # Reemplazar la capa final
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)

    return model.to(device)


model_demo = crear_modelo_resnet18()
print(model_demo.fc)
print("\nParámetros totales:", sum(p.numel() for p in model_demo.parameters()))
print("Parámetros entrenables:", sum(p.numel() for p in model_demo.parameters() if p.requires_grad))

## 3. Configuración del entrenamiento

- **Función de pérdida:** `CrossEntropyLoss`. Es la elección estándar para clasificación
  multiclase. No usamos focal loss porque, si bien hay desbalance de clases (Water tiene
  ~15x más ejemplos que Fairy), el desbalance no es extremo y CrossEntropy ya funciona
  razonablemente bien combinado con el fine-tuning de un modelo preentrenado.
- **Optimizador:** se prueban SGD (con y sin grupos de parámetros), AdamW y SGD con momentum.
- **Scheduler:** se prueba `ReduceLROnPlateau` en una de las configuraciones.
- **Batch size:** 32 (definido en `dataset_loader.py`).
- **Épocas:** entre 8 y 10 según el experimento.
- **Hardware:** Google Colab con GPU T4. Cada experimento tarda entre 3 y 8 minutos
  aproximadamente.

In [ ]:
def evaluate_model(model, loader, loss_fn, device, return_preds=False):
    """Evalúa el modelo en un DataLoader. Devuelve loss promedio y accuracy.
    Si return_preds=True, también devuelve las predicciones y etiquetas reales."""
    model.eval()
    L, N, correct, total = 0.0, 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            y_hat = model(X)
            l = loss_fn(y_hat, y)

            L += l.sum().item()
            N += l.numel()

            preds = y_hat.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.numel()

            if return_preds:
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(y.cpu().tolist())

    val_loss = L / N
    val_acc = correct / total

    if return_preds:
        return val_loss, val_acc, all_preds, all_labels
    return val_loss, val_acc

In [ ]:
def train_fine_tuning(model, learning_rate, num_epochs=5,
                       param_group=True, optimizer_name="sgd", scheduler_name=None):
    """
    Entrena un modelo y devuelve un historial con train_loss, val_loss y val_acc por época.

    optimizer_name: 'sgd' | 'sgd_momentum' | 'adam' | 'adamw'
    param_group: si True (solo aplica a 'sgd'), usa un learning rate 10x mayor para 'fc'
    scheduler_name: None | 'plateau'
    """
    loss_fn = nn.CrossEntropyLoss(reduction="none")

    if optimizer_name == "sgd":
        if param_group:
            params_1x = [
                param for name, param in model.named_parameters()
                if name not in ["fc.weight", "fc.bias"]
            ]
            optimizer = torch.optim.SGD(
                [
                    {"params": params_1x},
                    {"params": model.fc.parameters(), "lr": learning_rate * 10}
                ],
                lr=learning_rate, weight_decay=0.001
            )
        else:
            optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=0.001)

    elif optimizer_name == "sgd_momentum":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=0.001)

    elif optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.001)

    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    else:
        raise ValueError("Optimizador no reconocido")

    if scheduler_name == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    else:
        scheduler = None

    model.to(device)
    historial = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(num_epochs):
        model.train()
        L, N = 0.0, 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            l = loss_fn(model(X), y)

            optimizer.zero_grad()
            l.sum().backward()
            optimizer.step()

            L += l.sum().item()
            N += l.numel()

        train_loss = L / N
        val_loss, val_acc = evaluate_model(model, val_loader, loss_fn, device)

        if scheduler is not None:
            scheduler.step(val_loss)

        historial["train_loss"].append(train_loss)
        historial["val_loss"].append(val_loss)
        historial["val_acc"].append(val_acc)

        print(f"epoch {epoch + 1}/{num_epochs} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val acc {val_acc:.4f}")

    return historial

## 4. Experimentación

Se prueban 4 configuraciones distintas, variando optimizador, learning rate, scheduler y
estrategia de grupos de parámetros.

### Configuración 1 — SGD con learning rate diferenciado para `fc`

In [ ]:
model1 = crear_modelo_resnet18()
config_1 = train_fine_tuning(
    model1, learning_rate=5e-5, num_epochs=10,
    param_group=True, optimizer_name="sgd", scheduler_name=None
)

### Configuración 2 — SGD sin grupos de parámetros (mismo LR para toda la red)

In [ ]:
model2 = crear_modelo_resnet18()
config_2 = train_fine_tuning(
    model2, learning_rate=5e-5, num_epochs=10,
    param_group=False, optimizer_name="sgd", scheduler_name=None
)

### Configuración 3 — AdamW + ReduceLROnPlateau

In [ ]:
model3 = crear_modelo_resnet18()
config_3 = train_fine_tuning(
    model3, learning_rate=1e-4, num_epochs=8,
    param_group=False, optimizer_name="adamw", scheduler_name="plateau"
)

### Configuración 4 — SGD con momentum

In [ ]:
model4 = crear_modelo_resnet18()
config_4 = train_fine_tuning(
    model4, learning_rate=1e-4, num_epochs=10,
    param_group=False, optimizer_name="sgd_momentum", scheduler_name=None
)

### Tabla comparativa de experimentos

In [ ]:
import pandas as pd

tabla_experimentos = pd.DataFrame({
    "Configuración": [
        "Config 1 - SGD (LR diferenciado)",
        "Config 2 - SGD (LR único)",
        "Config 3 - AdamW + scheduler",
        "Config 4 - SGD + momentum"
    ],
    "Optimizador": ["SGD", "SGD", "AdamW", "SGD (momentum=0.9)"],
    "Learning rate": ["5e-5 / fc: 5e-4", "5e-5", "1e-4", "1e-4"],
    "Scheduler": ["No", "No", "ReduceLROnPlateau", "No"],
    "Épocas": [10, 10, 8, 10],
    "Mejor val accuracy": [
        max(config_1["val_acc"]),
        max(config_2["val_acc"]),
        max(config_3["val_acc"]),
        max(config_4["val_acc"]),
    ],
    "Menor val loss": [
        min(config_1["val_loss"]),
        min(config_2["val_loss"]),
        min(config_3["val_loss"]),
        min(config_4["val_loss"]),
    ],
})

tabla_experimentos

### Qué se aprendió de cada experimento

- **Config 1 (SGD, LR diferenciado):** dar un learning rate 10x mayor a la capa `fc`
  permite que la única capa entrenable converja más rápido, mientras el resto de la red
  se mantiene estable. Sirvió como primera referencia razonable.
- **Config 2 (SGD, LR único):** usar el mismo LR bajo (5e-5) para `fc` resultó en una
  convergencia más lenta, ya que esa capa parte de pesos aleatorios y necesita pasos más
  grandes para aprender. Esto confirmó que diferenciar el LR (Config 1) es una buena idea.
- **Config 3 (AdamW + scheduler):** Adam adapta el learning rate por parámetro, por lo que
  convergió más rápido en menos épocas. El scheduler `ReduceLROnPlateau` ayudó a afinar
  el entrenamiento cuando el val_loss dejó de mejorar.
- **Config 4 (SGD + momentum):** el momentum aceleró la convergencia respecto a SGD plano,
  pero con LR=1e-4 mostró más oscilación en el val_loss entre épocas.

**Decisión:** en base a la tabla comparativa, se elige la configuración con mayor
`val_accuracy` como modelo final (selección automática en la celda siguiente).

In [ ]:
configs = {
    "Config 1": (model1, config_1),
    "Config 2": (model2, config_2),
    "Config 3": (model3, config_3),
    "Config 4": (model4, config_4),
}

best_name = max(configs, key=lambda k: max(configs[k][1]["val_acc"]))
best_model, best_history = configs[best_name]

print(f"Configuración elegida: {best_name}")
print(f"Mejor val accuracy: {max(best_history['val_acc']):.4f}")

## 5. Evaluación del modelo elegido

### Curvas de pérdida y accuracy

In [ ]:
epochs_range = range(1, len(best_history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, best_history["train_loss"], label="Train loss", marker="o")
axes[0].plot(epochs_range, best_history["val_loss"], label="Val loss", marker="o")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].set_title(f"Curva de pérdida - {best_name}")
axes[0].legend()

axes[1].plot(epochs_range, best_history["val_acc"], label="Val accuracy", color="green", marker="o")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].set_title(f"Accuracy de validación - {best_name}")
axes[1].legend()

plt.tight_layout()
plt.show()

**Análisis de overfitting / underfitting:**

Como solo se entrena la capa `fc` (el resto de la red está congelada), el riesgo de
overfitting es bajo: el modelo tiene relativamente pocos parámetros entrenables en
comparación con el tamaño del dataset. Si se observa que `train_loss` sigue bajando
mientras `val_loss` se estanca o sube, sería indicio de overfitting; en ese caso se podría
aumentar el `weight_decay`, agregar más data augmentation o reducir el learning rate. Si
ambas curvas (train y val) siguen descendiendo al final del entrenamiento sin estancarse,
es indicio de underfitting y convendría entrenar más épocas o descongelar alguna capa
adicional.

### Métricas finales sobre el conjunto de TEST

In [ ]:
loss_fn = nn.CrossEntropyLoss(reduction="none")
test_loss, test_acc, test_preds, test_labels = evaluate_model(
    best_model, test_loader, loss_fn, device, return_preds=True
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print()
print("Reporte de clasificación (precision, recall, F1 por clase):")
print(classification_report(
    test_labels, test_preds,
    target_names=classes, digits=3, zero_division=0
))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(10, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title(f"Matriz de confusión - {best_name} (test)")
plt.tight_layout()
plt.show()

### Análisis de errores

Se muestran ejemplos del conjunto de test que fueron mal clasificados, junto con la
etiqueta real y la predicha.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    t = tensor.clone()
    for i, (m, s) in enumerate(zip(mean, std)):
        t[i] = t[i] * s + m
    return t.clamp(0, 1)


# Encontrar índices de errores recorriendo el test_loader (sin shuffle, mismo orden que test_dataset)
wrong_indices = [i for i, (p, l) in enumerate(zip(test_preds, test_labels)) if p != l]
print(f"Total de errores en test: {len(wrong_indices)} / {len(test_labels)} "
      f"({len(wrong_indices)/len(test_labels)*100:.1f}%)")

n_show = min(8, len(wrong_indices))
sample_wrong = random.sample(wrong_indices, n_show)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for ax, idx in zip(axes, sample_wrong):
    image, label = test_dataset[idx]
    img = denormalize(image).permute(1, 2, 0).numpy()

    real = idx_to_label[test_labels[idx]]
    pred = idx_to_label[test_preds[idx]]

    ax.imshow(img)
    ax.set_title(f"Real: {real}\nPred: {pred}", fontsize=9)
    ax.axis("off")

for ax in axes[n_show:]:
    ax.axis("off")

plt.suptitle("Ejemplos mal clasificados (test)", fontsize=13)
plt.tight_layout()
plt.show()

**Posibles causas de los errores observados:**

- **Clases visualmente similares:** tipos como Ground/Rock, o Bug/Grass, comparten
  colores y formas (tonos marrones/verdes), lo que genera confusión entre el modelo.
- **Pokémon con múltiples colores:** muchos Pokémon tienen colores que asociamos con
  varios tipos distintos (ej. un Pokémon verde y marrón podría confundirse entre Grass,
  Bug o Ground), aunque su `type1` real sea otro.
- **Desbalance de clases:** las clases minoritarias (Ice, Fairy, Dragon, Ghost) tienen
  pocos ejemplos de entrenamiento, por lo que el modelo tiene menos oportunidades de
  aprender sus características distintivas y tiende a confundirlas con clases mayoritarias
  (Water, Normal).
- **Calidad/variedad de las imágenes:** el dataset mezcla sprites 2D, artes oficiales y
  renders 3D, ahora todos con fondo negro uniforme, lo que reduce el ruido visual del fondo
  pero no elimina las diferencias de estilo entre imágenes.

## 6. Guardado del modelo final

In [ ]:
os.makedirs("dev", exist_ok=True)

torch.save(best_model.state_dict(), "dev/modelo.pth")

# Mapa de clases, necesario para la app de inferencia (entrega 4)
with open("dev/class_to_idx.json", "w") as f:
    json.dump(label_to_idx, f, ensure_ascii=False, indent=2)

print(f"Modelo guardado en dev/modelo.pth (configuración: {best_name})")
print(f"Mapa de clases guardado en dev/class_to_idx.json")
print(f"\nAccuracy final en test: {test_acc:.4f}")

## 7. Resumen final

In [ ]:
print("="*55)
print("RESUMEN - ENTRENAMIENTO Y SELECCIÓN DEL MODELO")
print("="*55)
print(f"Modelo base: ResNet18 (preentrenada en ImageNet)")
print(f"Estrategia: capas convolucionales congeladas, solo se entrena 'fc'")
print(f"Configuraciones probadas: 4")
print(f"Configuración elegida: {best_name}")
print(f"Mejor val accuracy: {max(best_history['val_acc']):.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Pesos guardados en: dev/modelo.pth")
print("="*55)